In [1]:
# required imports for plotting data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
cust_df = pd.read_csv("customers.csv")
order_df = pd.read_csv("orders.csv")

In [3]:
order_df.info()
# looks like a join table between order and customer 

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   OrderID     100 non-null    str    
 1   CustomerID  99 non-null     str    
 2   Date        100 non-null    str    
 3   Item        100 non-null    str    
 4   Category    100 non-null    str    
 5   Qty         99 non-null     float64
 6   Cost        100 non-null    str    
dtypes: float64(1), str(6)
memory usage: 3.2 KB


In [4]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CustomerID  40 non-null     str  
 1   Name        40 non-null     str  
 2   Location    40 non-null     str  
 3   Province    40 non-null     str  
 4   Type        39 non-null     str  
 5   Since       40 non-null     str  
dtypes: str(6)
memory usage: 1.0 KB


In [5]:
cust_df.head()

,CustomerID,Name,Location,Province,Type,Since
0,C0001,Amina Yusuf,Grande Prairie,Alberta,Standard,2020-12-29
1,C0002,Mateo Tremblay,Leduc,Alberta,VIP,2021-07-03
2,C0003,Priya Nair,Grande Prairie,Alberta,Standard,2019-07-01
3,C0004,Liam Chen,Sherwood Park,AB,Standard,2024-11-14
4,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27


In [6]:
# Checking cust_df for duplicates
cust_df.duplicated().sum()

np.int64(0)

In [7]:
# I assume that location, Province, Type and Since are not unique to customer
# That leaves Name and Customer ID as sources for duplicates
cust_df.duplicated(subset = ["Name"]).sum()

np.int64(1)

In [8]:
# I expect CustomerID isn't going to have duplicates. The assumption is that this value is computer generated and enforced as unique.
cust_df.duplicated(subset = ["CustomerID"]).sum()

np.int64(0)

In [9]:
# This leaves name as a source of duplicates
cust_df["Name"].value_counts()

Name
Jordan Lee         2
Amina Yusuf        1
Mateo Tremblay     1
Priya Nair         1
Liam Chen          1
Sofia Kowalski     1
Noah Cardinal      1
Fatima El-Sayed    1
Ethan Nguyen       1
Maya Desjarlais    1
Lucas Martin       1
Zahra Rahman       1
Owen McLeod        1
Anika Patel        1
Gabriel Wong       1
Leila Haddad       1
Jack Wilson        1
Nia Okafor         1
Xavier Gagnon      1
Mei Lin            1
Caleb Thompson     1
Samira Farouk      1
Theo Johnson       1
Ivy Park           1
Ravi Singh         1
Elena Popescu      1
Marcus Bird        1
Hana Kim           1
Julien Roy         1
Amara Mensah       1
Dylan Anderson     1
Layla Ali          1
Benito Garcia      1
Arjun Rao          1
Chloe Dubois       1
Malik Brooks       1
Yuna Sato          1
Cole Erickson      1
Aaliyah Khan       1
Name: count, dtype: int64

In [10]:
# checking the duplicate name to decide if the record needs to be dropped
cust_df[cust_df["Name"] == "Jordan Lee"]

,CustomerID,Name,Location,Province,Type,Since
32,C0033,Jordan Lee,Medicine Hat,Alberta,Premium,2019-10-13
39,C0040,Jordan Lee,Spruce Grove,Alberta,Premium,2023-01-13


Here's the dupe. Its possible this person was signed up twice, once per location.

This isn't an easy call, as two people can have the same first & last name

In [11]:
cust_df = cust_df.drop_duplicates(subset = ["Name"])

In [12]:
# this will have a knock on effect for the orders, we will need to replace C0040  with C0033 in the order table
order_df[order_df["CustomerID"] == "C0040"]
# luckily there are none, this rienforces my confidence in this removal

,OrderID,CustomerID,Date,Item,Category,Qty,Cost


In [13]:
order_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   OrderID     100 non-null    str    
 1   CustomerID  99 non-null     str    
 2   Date        100 non-null    str    
 3   Item        100 non-null    str    
 4   Category    100 non-null    str    
 5   Qty         99 non-null     float64
 6   Cost        100 non-null    str    
dtypes: float64(1), str(6)
memory usage: 3.2 KB


In [14]:
order_df.head()

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
0,O1001,C0034,2026-08-01,Rechargeable Light Set,Lights,2.0,79.99
1,O1001,C0034,2026-08-01,Trail Helmet,Helmets,1.0,89.99
2,O1001,C0034,2026-08-01,Bike Multi-Tool,Tools,1.0,34.99
3,O1002,C0037,2026-05-02,Commuter Helmet,Helmets,1.0,69.99
4,O1002,C0037,2026-05-02,Rear Safety Light,Lights,1.0,24.99


In [15]:
# checking order df for dupes
order_df.duplicated().sum()

np.int64(1)

In [16]:
# examining an instance of the duplicated record
order_df[order_df.duplicated()]

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
99,O1004,C0005,2026-01-30,Road Tire 700x28,Tires,1.0,64.99


In [17]:
# My assumption is that a record only becomes unique with a combination of OrderId and Item
# a customer can have the same order associated with them many times for each different item on that order
# but an item instance shouldn't be associated with the same order more than once.
# Multiple of the same item on one order should be tracked by qty instead.
# this should confirm if there are any other dupes
order_df[(order_df["OrderID"] == "O1004") & (order_df["Item"] == "Road Tire 700x28")]

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
10,O1004,C0005,2026-01-30,Road Tire 700x28,Tires,1.0,64.99
99,O1004,C0005,2026-01-30,Road Tire 700x28,Tires,1.0,64.99


In [18]:
order_df = order_df.drop_duplicates()

In [19]:
order_df.isna().sum()

OrderID       0
CustomerID    1
Date          0
Item          0
Category      0
Qty           1
Cost          0
dtype: int64

In [20]:
# check out rows with at least one NaN value
order_df[order_df.isna().any(axis=1)]


,OrderID,CustomerID,Date,Item,Category,Qty,Cost
6,O1003,C0019,2026-08-13,U-Lock,Locks,NaN,59.99
15,O1006,NaN,2026-08-21,Front LED Light,Lights,1.0,39.99


In [21]:
order_df[order_df["Item"] == "phone mount"]

,OrderID,CustomerID,Date,Item,Category,Qty,Cost


In [22]:
# how much does the U-Lock usually cost?
order_df[order_df["Item"] == "U-Lock"]

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
6,O1003,C0019,2026-08-13,U-Lock,Locks,NaN,59.99
30,O1011,C0005,2026-03-19,U-Lock,Locks,1.0,59.99
42,O1015,C0039,2026-07-28,U-Lock,Locks,1.0,59.99
82,O1028,C0030,2026-04-07,U-Lock,Locks,1.0,59.99


Deleting a financial record is likely a bad idea.
# Missing Quantity
We can replace this missing value with 1, a
# Missing CustomerID
This will be replaced with "C0014", as the order has other items on it all associated with that CustomerID
I will assume only one customer can be associated with an order.

In [23]:
order_df["CustomerID"] = order_df["CustomerID"].fillna("C0014")
# this wouldn't work as well if there was more than one missing CustomerID

In [24]:
order_df["Qty"] = order_df["Qty"].fillna(1)
# this also wouldnt work too well if there were more missing quantities

In [25]:
order_df.isna().sum()

OrderID       0
CustomerID    0
Date          0
Item          0
Category      0
Qty           0
Cost          0
dtype: int64

In [26]:
# Check both tables for null values
cust_df.isna().sum()

CustomerID    0
Name          0
Location      0
Province      0
Type          1
Since         0
dtype: int64

In [27]:
# check out rows with at least one NaN value
cust_df[cust_df.isna().any(axis=1)]

,CustomerID,Name,Location,Province,Type,Since
16,C0017,Nia Okafor,Airdrie,Alberta,NaN,2024-07-10


In [28]:
# what is this column?
cust_df["Type"].unique()

<StringArray>
['Standard', 'VIP', 'Premium', 'standard', nan, 'Std']
Length: 6, dtype: str

# Missing Type
No obvious way to determine what this Type value would have been.
Most of this record is still useful
This can be replaced by setting it to the mode, 

In [29]:
cust_df["Type"].mode()

0    Standard
Name: Type, dtype: str

In [30]:
cust_df["Type"] = cust_df["Type"].fillna("Standard")


In [31]:
cust_df.isna().sum()

CustomerID    0
Name          0
Location      0
Province      0
Type          0
Since         0
dtype: int64

In [32]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CustomerID  39 non-null     str  
 1   Name        39 non-null     str  
 2   Location    39 non-null     str  
 3   Province    39 non-null     str  
 4   Type        39 non-null     str  
 5   Since       39 non-null     str  
dtypes: str(6)
memory usage: 1008.0 bytes


To keep consistent with the current names, these will be set to PascalCase.

Name could be a little more descriptive, CustomerName is better

What is Location? City is more descriptive

What is Type? CustomerTier is more descriptive

Since is a little vague. Changing to DateAdded

In [33]:
cust_df = cust_df.rename(columns = {"Name": "CustomerName", "Location": "City", "Type": "CustomerTeir", "Since": "DateAdded"})
cust_df.head(0)

,CustomerID,CustomerName,City,Province,CustomerTeir,DateAdded


In [34]:
order_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   OrderID     99 non-null     str    
 1   CustomerID  99 non-null     str    
 2   Date        99 non-null     str    
 3   Item        99 non-null     str    
 4   Category    99 non-null     str    
 5   Qty         99 non-null     float64
 6   Cost        99 non-null     str    
dtypes: float64(1), str(6)
memory usage: 3.2 KB


In [35]:
# Is cost per item or the total?
order_df[order_df["Item"] == "Rechargeable Light Set"]

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
0,O1001,C0034,2026-08-01,Rechargeable Light Set,Lights,2.0,79.99
22,O1008,C0036,2026-06-08,Rechargeable Light Set,Lights,1.0,79.99
92,O1032,C0011,2026-03-10,Rechargeable Light Set,Lights,2.0,79.99
97,O1034,C0006,2026-01-08,Rechargeable Light Set,Lights,1.0,79.99


Sticking with PascalCase for naming these.

Item is a little generic. Replacing it with 'Product'.

Category is also generic. Replace with 'ProductCategory'.

Date could be more descriptive, 'OrderDate' is better.

Cost isn't sufficiently. Replacing it with 'UnitPrice'.

In [36]:
order_df = order_df.rename(columns = {"Item": "Product", "Category": "ProductCategory", "Date": "OrderDate", "Cost": "UnitPrice"})
order_df.head(0)

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice


# Transformations

In [37]:

# To make a column for total, cost needs to be a float
# Some of these values have '$' in them
order_df["UnitPrice"] = order_df["UnitPrice"].str.replace("$", "")
order_df["UnitPrice"] = pd.to_numeric(order_df["UnitPrice"])
order_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          99 non-null     str    
 1   CustomerID       99 non-null     str    
 2   OrderDate        99 non-null     str    
 3   Product          99 non-null     str    
 4   ProductCategory  99 non-null     str    
 5   Qty              99 non-null     float64
 6   UnitPrice        99 non-null     float64
dtypes: float64(2), str(5)
memory usage: 3.6 KB


In [38]:
# Set a new column "Total" by multiplying Qty and UnitPrice
order_df["Total"] = order_df["Qty"] * order_df["UnitPrice"]
order_df.head()

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total
0,O1001,C0034,2026-08-01,Rechargeable Light Set,Lights,2.0,79.99,159.98
1,O1001,C0034,2026-08-01,Trail Helmet,Helmets,1.0,89.99,89.99
2,O1001,C0034,2026-08-01,Bike Multi-Tool,Tools,1.0,34.99,34.99
3,O1002,C0037,2026-05-02,Commuter Helmet,Helmets,1.0,69.99,69.99
4,O1002,C0037,2026-05-02,Rear Safety Light,Lights,1.0,24.99,24.99


Next up is normalizing the data

It helps to check for outliers

In [39]:
# It is important to check for outliers before scaling
odf_q1, odf_q3 = order_df["UnitPrice"].quantile([0.25, 0.75])
print(f"Quartile 1: {str(odf_q1)} \nQuartile 2: {str(odf_q3)}")
odf_iqr = odf_q3 - odf_q1
print(f"IQR: {odf_iqr}")
odf_lowerb = odf_q1 - 1.5*odf_iqr
odf_upperb = odf_q3 + 1.5*odf_iqr
print(f"Lower bound: {odf_lowerb} \nUpper bound: {odf_upperb}")

# the lower bound was negative, there shouldnt be negative prices, but never say never
order_df[order_df["UnitPrice"] < 1]


Quartile 1: 39.99 
Quartile 2: 89.99
IQR: 49.99999999999999
Lower bound: -35.009999999999984 
Upper bound: 164.98999999999998


,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total


In [40]:
# since there are no negative prices, only need to check the upper bound
outliers = order_df[order_df["UnitPrice"] > odf_upperb]

outliers.head()

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total
11,O1004,C0005,2026-01-30,City Commuter Bike,Bicycles,1.0,899.99,899.99
31,O1011,C0005,2026-03-19,Hybrid Fitness Bike,Bicycles,1.0,1099.99,1099.99
33,O1012,C0010,2026-02-15,City Commuter Bike,Bicycles,-3.0,899.99,-2699.97
37,O1013,C0008,2026-08-07,Hybrid Fitness Bike,Bicycles,1.0,1099.99,1099.99
41,O1014,C0006,2026-09-24,Trail Mountain Bike,Bicycles,1.0,1499.99,1499.99


In [41]:
# There are quite a few outliers. More than 15% of values are Outliers 
# This does reveal an issue with Qty, where a negative qty has appeared
# Taking a cleaning detour to remove this issue
# I will ASSUME that these negative quantities only need to be flipped to positive to be correct
order_df["Qty"] = abs(order_df["Qty"])
# and have to fix the total with it
order_df["Total"] = abs(order_df["Total"])
order_df.loc[33]

OrderID                         O1012
CustomerID                      C0010
OrderDate                  2026-02-15
Product            City Commuter Bike
ProductCategory              Bicycles
Qty                               3.0
UnitPrice                      899.99
Total                         2699.97
Name: 33, dtype: object

In [42]:
# min-max is affected by outliers leaving most values to be quite tiny.
# To account for a strong presense of outliers, we can use Robust scaling
# Robust scaling uses the median as the measure of central tendency to scale against,
# and checks how many interQuartile ranges a value is from the median.
# This should be highly resistant to outliers
# To calculate: RobustScaleValue = (value - median) / iqr
order_df["ScaledUnitPrice"] = (order_df["UnitPrice"] - order_df["UnitPrice"].median()) / odf_iqr
order_df.head(10)

# Keep in mind this is scaled based on every item sold in every order, so some items may represent more than others.
# A change in the volume of sales will change the scaled unit price

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total,ScaledUnitPrice
0,O1001,C0034,2026-08-01,Rechargeable Light Set,Lights,2.0,79.99,159.98,0.2
1,O1001,C0034,2026-08-01,Trail Helmet,Helmets,1.0,89.99,89.99,0.4
2,O1001,C0034,2026-08-01,Bike Multi-Tool,Tools,1.0,34.99,34.99,-0.7
3,O1002,C0037,2026-05-02,Commuter Helmet,Helmets,1.0,69.99,69.99,0.0
4,O1002,C0037,2026-05-02,Rear Safety Light,Lights,1.0,24.99,24.99,-0.9
5,O1002,C0037,2026-05-02,Folding Lock,Locks,3.0,119.99,359.97,1.0
6,O1003,C0019,2026-08-13,U-Lock,Locks,1.0,59.99,59.99,-0.2
7,O1003,C0019,2026-08-13,Youth Helmet,Helmets,1.0,49.99,49.99,-0.4
8,O1003,C0019,2026-08-13,Phone Mount,Accessories,1.0,39.99,39.99,-0.6
9,O1004,C0005,2026-01-30,Front LED Light,Lights,3.0,39.99,119.97,-0.6


# The Date

In [43]:
# both tables appear to use a year, month, day date system
# Trying out a to_datetime method to see if it will convert nicely.
try:
    pd.to_datetime(order_df["OrderDate"])
except:
    print("The dates did not convert nicely")

The dates did not convert nicely


In [44]:
# Checking out dates that dont match the expected format
order_df[~order_df["OrderDate"].str.contains("-")]

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total,ScaledUnitPrice
20,O1007,C0026,02/09/2026,Cable Lock,Locks,2.0,29.99,59.98,-0.8


In [45]:
# not only does this date have '/' instead of '-', its unclear where the day and month are
# We can check this by seeing if this order had other items on it
order_df[order_df["CustomerID"] == "C0026"]

,OrderID,CustomerID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total,ScaledUnitPrice
18,O1007,C0026,2026-02-09,Rear Safety Light,Lights,1.0,24.99,24.99,-0.9
19,O1007,C0026,2026-02-09,Puncture-Resistant Tire,Tires,1.0,84.99,84.99,0.3
20,O1007,C0026,02/09/2026,Cable Lock,Locks,2.0,29.99,59.98,-0.8


In [46]:
# Use mixed to convert the variable date formats
order_df["OrderDate"] = pd.to_datetime(order_df["OrderDate"], format="mixed")

order_df["OrderDate"][20]

Timestamp('2026-02-09 00:00:00')

In [47]:
# We got lucky and it parsed the month the correct way
order_df["OrderDate"].info()

<class 'pandas.Series'>
RangeIndex: 99 entries, 0 to 98
Series name: OrderDate
Non-Null Count  Dtype         
--------------  -----         
99 non-null     datetime64[us]
dtypes: datetime64[us](1)
memory usage: 864.0 bytes


In [48]:
# Check if the customer table dates will convert nicely
try:
    cust_df["DateAdded"] = pd.to_datetime(cust_df["DateAdded"])
except:
    print("The dates did not convert nicely")

In [49]:
# This appeared to work
cust_df["DateAdded"].info()

<class 'pandas.Series'>
RangeIndex: 39 entries, 0 to 38
Series name: DateAdded
Non-Null Count  Dtype         
--------------  -----         
39 non-null     datetime64[us]
dtypes: datetime64[us](1)
memory usage: 384.0 bytes


# Joining

In [50]:
order_cust_df = cust_df.merge(order_df, on="CustomerID", how="outer", indicator=True)
order_cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CustomerID       114 non-null    str           
 1   CustomerName     113 non-null    str           
 2   City             113 non-null    str           
 3   Province         113 non-null    str           
 4   CustomerTeir     113 non-null    str           
 5   DateAdded        113 non-null    datetime64[us]
 6   OrderID          99 non-null     str           
 7   OrderDate        99 non-null     datetime64[us]
 8   Product          99 non-null     str           
 9   ProductCategory  99 non-null     str           
 10  Qty              99 non-null     float64       
 11  UnitPrice        99 non-null     float64       
 12  Total            99 non-null     float64       
 13  ScaledUnitPrice  99 non-null     float64       
 14  _merge           114 non-null    category      
dtype

In [51]:
# lets see those null rows
order_cust_df[order_cust_df.isna().any(axis=1)]


,CustomerID,CustomerName,City,Province,CustomerTeir,DateAdded,OrderID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total,ScaledUnitPrice,_merge
3,C0002,Mateo Tremblay,Leduc,Alberta,VIP,2021-07-03,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,C0003,Priya Nair,Grande Prairie,Alberta,Standard,2019-07-01,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
5,C0004,Liam Chen,Sherwood Park,AB,Standard,2024-11-14,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
20,C0007,Fatima El-Sayed,Red Deer,Alberta,Standard,2023-11-20,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
34,C0009,Maya Desjarlais,St. Albert,Alberta,Premium,2021-02-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
40,C0012,Owen McLeod,Sherwood Park,Alberta,Standard,2024-11-29,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
49,C0015,Leila Haddad,Lethbridge,Alberta,VIP,2018-10-16,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
68,C0021,Samira Farouk,Red Deer,Alberta,Standard,2020-07-11,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
71,C0023,Ivy Park,Leduc,Alberta,Premium,2021-04-02,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
72,C0024,Ravi Singh,Airdrie,Alberta,Premium,2019-12-03,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [52]:
# customers who never purchased anything... and one product purchased by someone not recorded by the customer data
order_cust_df.head(10)

,CustomerID,CustomerName,City,Province,CustomerTeir,DateAdded,OrderID,OrderDate,Product,ProductCategory,Qty,UnitPrice,Total,ScaledUnitPrice,_merge
0,C0001,Amina Yusuf,Grande Prairie,Alberta,Standard,2020-12-29,O1022,2026-08-22,Folding Lock,Locks,1.0,119.99,119.99,1.0,both
1,C0001,Amina Yusuf,Grande Prairie,Alberta,Standard,2020-12-29,O1022,2026-08-22,Youth Helmet,Helmets,1.0,49.99,49.99,-0.4,both
2,C0001,Amina Yusuf,Grande Prairie,Alberta,Standard,2020-12-29,O1022,2026-08-22,City Commuter Bike,Bicycles,2.0,899.99,1799.98,16.6,both
3,C0002,Mateo Tremblay,Leduc,Alberta,VIP,2021-07-03,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,C0003,Priya Nair,Grande Prairie,Alberta,Standard,2019-07-01,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
5,C0004,Liam Chen,Sherwood Park,AB,Standard,2024-11-14,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27,O1004,2026-01-30,Front LED Light,Lights,3.0,39.99,119.97,-0.6,both
7,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27,O1004,2026-01-30,Road Tire 700x28,Tires,1.0,64.99,64.99,-0.1,both
8,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27,O1004,2026-01-30,City Commuter Bike,Bicycles,1.0,899.99,899.99,16.6,both
9,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27,O1011,2026-03-19,U-Lock,Locks,1.0,59.99,59.99,-0.2,both


In [53]:
order_totals_df = order_df.groupby(["CustomerID"]).agg({"Total": "sum" })
order_totals_df.head()

,Total
CustomerID,
C0001,1969.96
C0005,2264.92
C0006,2229.88
C0008,8179.84
C0010,2894.95


In [54]:
order_totals_df["Total"].mean()

np.float64(1460.74)

In [55]:
order_totals_df["Total"].median()

np.float64(569.91)

In [56]:
ot_q1, ot_q3 = order_totals_df["Total"].quantile([0.25, 0.75])
print(f"Quartile 1: {str(ot_q1)} \nQuartile 2: {str(ot_q3)}")
ot_iqr = ot_q3 - ot_q1
print(f"IQR: {ot_iqr}")
ot_lowerb = ot_q1 - 1.5*ot_iqr
ot_upperb = ot_q3 + 1.5*ot_iqr
print(f"Lower bound: {ot_lowerb} \nUpper bound: {ot_upperb}")

Quartile 1: 229.96999999999997 
Quartile 2: 2264.92
IQR: 2034.95
Lower bound: -2822.4550000000004 
Upper bound: 5317.345


In [57]:
# We solved the issue of negative orders earlier, only need to check for positive outliers
ot_outliers = order_totals_df[order_totals_df["Total"] > ot_upperb]

ot_outliers.head()

,Total
CustomerID,
C0008,8179.84


In [ ]:
# : )